In [ ]:
import numpy as np
import scipy.sparse as sp
import matplotlib.pyplot as plt
import pandas as pd
import os
import gc
import zipfile
from numba import njit
from tqdm import tqdm

# ==========================================
# 1. ネットワーク生成エンジン (Numba JIT)
# ==========================================
@njit
def build_ba_edges_fast(N, m):
    """単一のBarabási-Albertモデルエッジを生成"""
    num_edges = 2 * m * N
    row = np.empty(num_edges, dtype=np.int32)
    col = np.empty(num_edges, dtype=np.int32)
    targets = np.empty(num_edges, dtype=np.int32)

    idx = 0; target_idx = 0

    for i in range(m):
        for j in range(m):
            if i != j:
                row[idx] = i; col[idx] = j; idx += 1
                targets[target_idx] = i; target_idx += 1

    chosen = np.empty(m, dtype=np.int32)
    for i in range(m, N):
        c_idx = 0
        while c_idx < m:
            r = np.random.randint(0, target_idx)
            node = targets[r]
            already = False
            for k in range(c_idx):
                if chosen[k] == node:
                    already = True
                    break
            if not already:
                chosen[c_idx] = node
                c_idx += 1

        for k in range(m):
            target = chosen[k]
            row[idx] = i; col[idx] = target; idx += 1
            row[idx] = target; col[idx] = i; idx += 1
            targets[target_idx] = i; target_idx += 1
            targets[target_idx] = target; target_idx += 1

    return row[:idx], col[:idx]

def build_clustered_network(N, m, weak_ties_ratio=0.002):
    """2つのスケールフリーコミュニティを弱い紐帯で結合したネットワーク"""
    N_A = N // 2
    N_B = N - N_A

    # Cluster A (Minority/Target)とCluster B (Majority)を独立に生成
    row_A, col_A = build_ba_edges_fast(N_A, m)
    row_B, col_B = build_ba_edges_fast(N_B, m)

    # Cluster BのIDをシフト
    row_B = row_B + N_A
    col_B = col_B + N_A

    # エッジの結合
    row = np.concatenate([row_A, row_B])
    col = np.concatenate([col_A, col_B])

    # 弱い紐帯 (Weak Ties) を追加
    num_weak_ties = max(1, int(N * weak_ties_ratio))
    wt_row = np.random.randint(0, N_A, num_weak_ties)
    wt_col = np.random.randint(N_A, N, num_weak_ties)

    row = np.concatenate([row, wt_row, wt_col])
    col = np.concatenate([col, wt_col, wt_row])

    data = np.ones(len(row), dtype=np.float32)
    A_sparse = sp.csr_matrix((data, (row, col)), shape=(N, N))

    # Cluster A のハブを特定
    deg_A = np.bincount(row_A, minlength=N_A)
    hub_A = np.argmax(deg_A)

    return A_sparse, N_A, hub_A

# ==========================================
# 2. シミュレーション関数 (カスケード・ダイナミクス)
# ==========================================
def run_cascade_simulation(N, A_sparse, N_A, hub_A, params):
    dt = params['dt']
    steps = params['steps']
    T_expose = params['T_expose']

    # 孤立ノード回避
    degrees = np.array(A_sparse.sum(axis=1)).flatten()
    degrees[degrees == 0] = 1.0

    # 乱数による初期値設定（初期値依存の完全排除）
    X = np.random.uniform(0, 0.2, N).astype(np.float32)
    D = np.random.uniform(0, 0.1, N).astype(np.float32)

    # ターゲット係数 v_i の設定 (Cluster A: 1.0, Cluster B: 0.3)
    v_target = np.full(N, 0.3, dtype=np.float32)
    v_target[:N_A] = 1.0

    history_D_A = []
    history_D_B = []

    b_val = 0.95 # 強い情報ウイルス
    b = np.zeros(N, dtype=np.float32)

    for step in tqdm(range(steps), desc=f"Simulating", leave=False):
        if step < T_expose:
            b[hub_A] = b_val
        else:
            b[hub_A] = 0.0 # ファクトチェック（暴露）

        # 同調圧力の局所伝播
        neighbor_sum = A_sparse.dot(X)
        P = neighbor_sum / degrees

        # SDE更新
        noise = np.random.normal(0, 1, N).astype(np.float32) * np.sqrt(dt)
        dX = (P + b - X) * dt + params['sigma'] * noise
        X_new = np.clip(X + dX, 0, 1)

        # 壊死と線維化（トポロジカル・ペナルティ）
        repair_capacity = np.exp(-params['gamma'] * D)
        dD = (params['k_break'] * v_target * X_new * (1 - D) -
              params['k_restore'] * (1 - X_new) * D * repair_capacity) * dt
        D_new = np.clip(D + dD, 0, 1)

        X = X_new
        D = D_new

        history_D_A.append(np.mean(D[:N_A]))
        history_D_B.append(np.mean(D[N_A:]))

    return history_D_A, history_D_B

# ==========================================
# 3. メイン実行スクリプト
# ==========================================
def main():
    N_list = [100, 1000, 10000, 100000, 1000000, 10000000]
    m = 2

    params = {
        'dt': 0.1,
        'steps': 150,      # カスケードを見るため長めに設定
        'T_expose': 80,    # フェイクへの暴露時間
        'sigma': 0.05,
        'k_break': 0.6,
        'k_restore': 0.1,
        'gamma': 6.0       # ペナルティをやや強く設定
    }

    output_dir = "tmfg_hypothesis2_results"
    os.makedirs(output_dir, exist_ok=True)
    summary_data = []

    for N in N_list:
        print(f"\n[{N} Nodes] ネットワーク生成中 (Clustered SBM-BA Model)...")
        try:
            A_sparse, N_A, hub_A = build_clustered_network(N, m)

            print(f"-> シミュレーション実行中 (カスケード追跡)")
            hist_D_A, hist_D_B = run_cascade_simulation(N, A_sparse, N_A, hub_A, params)

            del A_sparse
            gc.collect()

            # CSV出力
            df = pd.DataFrame({
                'TimeStep': np.arange(params['steps']),
                'D_ClusterA_Minority': hist_D_A,
                'D_ClusterB_Majority': hist_D_B
            })
            csv_path = os.path.join(output_dir, f'result_N{N}.csv')
            df.to_csv(csv_path, index=False)

            # グラフの出力
            plt.figure(figsize=(10, 6))
            plt.plot(df['TimeStep'], df['D_ClusterA_Minority'], 'r-', linewidth=3, label='Cluster A (Focal Infection)')
            plt.plot(df['TimeStep'], df['D_ClusterB_Majority'], 'b--', linewidth=2, label='Cluster B (Metastasis via Weak Ties)')
            plt.axvline(x=params['T_expose'], color='k', linestyle=':', label='Fake Exposed / Intervened')
            plt.title(f'Cascade of Topological Penalty: Multiple Organ Failure (N={N:,})')
            plt.xlabel('Time Step')
            plt.ylabel('Topological Defect (D)')
            plt.legend()
            plt.grid(True, alpha=0.3)
            png_path = os.path.join(output_dir, f'plot_N{N}.png')
            plt.savefig(png_path, dpi=300, bbox_inches='tight')
            plt.close()

            summary_data.append({'N': N, 'Final_D_A': hist_D_A[-1], 'Final_D_B': hist_D_B[-1]})

        except MemoryError:
            print(f"[警告] N={N} でメモリ不足 (OOM) が発生。以降の巨大Nはスキップします。")
            break

    # サマリーとZIP化
    if len(summary_data) > 0:
        df_sum = pd.DataFrame(summary_data)
        df_sum.to_csv(os.path.join(output_dir, 'summary_H2.csv'), index=False)

        plt.figure(figsize=(8, 6))
        plt.plot(df_sum['N'], df_sum['Final_D_A'], 'r-o', label='Cluster A (Origin)')
        plt.plot(df_sum['N'], df_sum['Final_D_B'], 'b--s', label='Cluster B (Systemic)')
        plt.xscale('log')
        plt.title('Macroscopic Scale vs Final Systemic Defect')
        plt.xlabel('Sample Size N (Log Scale)')
        plt.ylabel('Final Defect Level (Hysteresis)')
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.savefig(os.path.join(output_dir, 'summary_plot.png'), dpi=300, bbox_inches='tight')
        plt.close()

        zip_filename = 'TMFG_Hypothesis2_Results.zip'
        with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
            for root, _, files in os.walk(output_dir):
                for file in files:
                    zipf.write(os.path.join(root, file), arcname=file)

        print(f"\n🎉 完了: 結果を '{zip_filename}' に出力しました。ダウンロードしてご確認ください！")

if __name__ == "__main__":
    main()


[100 Nodes] ネットワーク生成中 (Clustered SBM-BA Model)...
-> シミュレーション実行中 (カスケード追跡)



[1000 Nodes] ネットワーク生成中 (Clustered SBM-BA Model)...
-> シミュレーション実行中 (カスケード追跡)



[10000 Nodes] ネットワーク生成中 (Clustered SBM-BA Model)...
-> シミュレーション実行中 (カスケード追跡)



[100000 Nodes] ネットワーク生成中 (Clustered SBM-BA Model)...
-> シミュレーション実行中 (カスケード追跡)



[1000000 Nodes] ネットワーク生成中 (Clustered SBM-BA Model)...
-> シミュレーション実行中 (カスケード追跡)



[10000000 Nodes] ネットワーク生成中 (Clustered SBM-BA Model)...
-> シミュレーション実行中 (カスケード追跡)



🎉 完了: 結果を 'TMFG_Hypothesis2_Results.zip' に出力しました。ダウンロードしてご確認ください！
